In [ ]:
import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from imblearn.combine import SMOTETomek

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report
)

from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
def load_train_test(train_path, test_path, target_col='is_fraud'):
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]

    X_test  = test_df.drop(columns=[target_col])
    y_test  = test_df[target_col]

    return X_train, y_train, X_test, y_test


In [ ]:
def preprocess_data(X_train, X_test, categorical_cols):
    numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
        ],
        remainder="drop"
    )

    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p  = preprocessor.transform(X_test)


    if hasattr(X_train_p, "toarray"):
        X_train_p = X_train_p.toarray()
    if hasattr(X_test_p, "toarray"):
        X_test_p = X_test_p.toarray()

    return X_train_p, X_test_p, preprocessor


In [ ]:
def apply_smote_tomek(X_train, y_train, random_state=42):
    smt = SMOTETomek(random_state=random_state)
    X_resampled, y_resampled = smt.fit_resample(X_train, y_train)
    return X_resampled, y_resampled


In [ ]:
def train_base_models(X_train_fit, y_train_fit, random_state=42):
    models = {
        "logistic_regression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=random_state
        ),
        "knn": KNeighborsClassifier(
            n_neighbors=7
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=400,
            max_depth=12,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=random_state
        ),
        "gaussian_nb": GaussianNB()
    }

    for name, model in models.items():
        model.fit(X_train_fit, y_train_fit)

    return models


In [ ]:
def pick_threshold_max_fbeta(y_true, y_prob, beta=2.0):
    precision, recall, thr = precision_recall_curve(y_true, y_prob)

    thr = np.append(thr, 1.0)

    beta2 = beta ** 2
    fbeta = (1 + beta2) * (precision * recall) / (beta2 * precision + recall + 1e-12)

    return float(thr[np.nanargmax(fbeta)])


In [ ]:
def threshold_aware_aggregation(models, X, thresholds, eps=1e-9):
    p_lr  = models["logistic_regression"].predict_proba(X)[:, 1]
    p_knn = models["knn"].predict_proba(X)[:, 1]
    p_rf  = models["random_forest"].predict_proba(X)[:, 1]
    p_nb  = models["gaussian_nb"].predict_proba(X)[:, 1]

    probs = np.column_stack([p_lr, p_knn, p_rf, p_nb])
    probs = np.clip(probs, eps, 1 - eps)

    t = np.array([
        thresholds["logistic_regression"],
        thresholds["knn"],
        thresholds["random_forest"],
        thresholds["gaussian_nb"]
    ])
    t = np.clip(t, eps, 1 - eps)

    weights = np.maximum(probs - t, 0.0) / (1.0 - t + eps)
    wsum = weights.sum(axis=1)

    fallback = probs.max(axis=1)

    agg = np.where(
        wsum > 0,
        (weights * probs).sum(axis=1) / (wsum + eps),
        fallback
    )

    return agg


In [ ]:
def evaluate_and_save_results(y_true, y_prob, threshold=0.5, json_path="results.json"):
    y_pred = (y_prob >= threshold).astype(int)

    roc_auc = roc_auc_score(y_true, y_prob)
    avg_precision = average_precision_score(y_true, y_prob)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    results = {
        "classification_report": report,
        "threshold_used_for_final_decision": float(threshold),
        "roc_auc": float(roc_auc),
        "average_precision": float(avg_precision),
        "confusion_matrix": {"labels": ["0", "1"], "matrix": cm.tolist()},
        "roc_curve": {"fpr": fpr.tolist(), "tpr": tpr.tolist()},
        "precision_recall_curve": {"precision": precision.tolist(), "recall": recall.tolist()},

    }

    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)

    return results


In [ ]:
# Load data
X_train, y_train, X_test, y_test = load_train_test(
    PROJECT_ROOT / "Data Understanding, Exploration & Feature Engineering/Dataset & Feature Engineered data/train.csv",
    PROJECT_ROOT / "Data Understanding, Exploration & Feature Engineering/Dataset & Feature Engineered data/test.csv",
    target_col="is_fraud"
)

categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

# Preprocess the data
X_train_p, X_test_p, preprocessor = preprocess_data(X_train, X_test, categorical_cols)

print("Train shape:", X_train_p.shape, "Test shape:", X_test_p.shape)
print("Fraud rate (train):", float(np.mean(y_train)))
print("Fraud rate (test) :", float(np.mean(y_test)))


Train shape: (239756, 43) Test shape: (59939, 43)
Fraud rate (train): 0.02206409850014181
Fraud rate (test) : 0.022055756685964063


In [ ]:
RANDOM_STATE = 42

# Validation split (only from training)
X_tr_p, X_val_p, y_tr, y_val = train_test_split(
    X_train_p, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=RANDOM_STATE
)

USE_SMOTE = True

if USE_SMOTE:
    X_fit, y_fit = apply_smote_tomek(X_tr_p, y_tr, random_state=RANDOM_STATE)
else:
    X_fit, y_fit = X_tr_p, y_tr

print("Before resampling:", pd.Series(y_tr).value_counts().to_dict())
print("After resampling :", pd.Series(y_fit).value_counts().to_dict())


In [ ]:
# Training base models
models = train_base_models(X_fit, y_fit, random_state=RANDOM_STATE)

# Predicting on val to learn each model's threshold
p_val_lr  = models["logistic_regression"].predict_proba(X_val_p)[:, 1]
p_val_knn = models["knn"].predict_proba(X_val_p)[:, 1]
p_val_rf  = models["random_forest"].predict_proba(X_val_p)[:, 1]
p_val_nb  = models["gaussian_nb"].predict_proba(X_val_p)[:, 1]

thresholds = {
    "logistic_regression": pick_threshold_max_fbeta(y_val, p_val_lr, beta=2.0),
    "knn": pick_threshold_max_fbeta(y_val, p_val_knn, beta=2.0),
    "random_forest": pick_threshold_max_fbeta(y_val, p_val_rf, beta=2.0),
    "gaussian_nb": pick_threshold_max_fbeta(y_val, p_val_nb, beta=2.0),
}

print("Per-model thresholds (from validation):")
for k, v in thresholds.items():
    print(f"  {k}: {v:.4f}")

# Threshold-aware aggregation on test
p_thresh_aware = threshold_aware_aggregation(models, X_test_p, thresholds)

FINAL_ENSEMBLE_THRESHOLD = 0.5

results = evaluate_and_save_results(
    y_test,
    p_thresh_aware,
    threshold=FINAL_ENSEMBLE_THRESHOLD,
    json_path="results/Baseline_model_with_threshold_aware_aggregation_results.json"
)

results["classification_report"]["1"]


In [ ]:
# ROC Curve
fpr = results["roc_curve"]["fpr"]
tpr = results["roc_curve"]["tpr"]
roc_auc = results["roc_auc"]

plt.figure()
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Threshold-Aware Aggregation")
plt.legend()
plt.grid(True)
plt.show()

# PR Curve
precision = results["precision_recall_curve"]["precision"]
recall = results["precision_recall_curve"]["recall"]
avg_precision = results["average_precision"]

plt.figure()
plt.plot(recall, precision, label=f"PR Curve (AP = {avg_precision:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve – Threshold-Aware Aggregation")
plt.legend()
plt.grid(True)
plt.show()

# Confusion Matrix
cm = np.array(results["confusion_matrix"]["matrix"])
labels = results["confusion_matrix"]["labels"]

plt.figure()
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap="winter")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix – Threshold-Aware Aggregation")
plt.show()
